# Agent Trust Verification with AG2 and TWZRD Agent Intel MCP

This notebook shows how to connect AG2 agents to the [TWZRD Agent Intel](https://intel.twzrd.xyz) MCP server
to score Solana AI agent wallets before authorizing x402 micropayments.

**TWZRD Agent Intel** is a zero-install MCP server accessible via streamable-HTTP:
- `score_agent(wallet)` — on-chain trust score 0–1 (free)
- `preflight_check(wallet)` — full pre-payment due diligence (free)
- `get_trust_receipt(wallet)` — signed trust receipt via HTTP 402 (paid)

No server to run locally. No API key required for the free tools.


## Install Dependencies


In [ ]:
# Install required packages
# pip install ag2 mcp nest_asyncio

## Imports and Setup


In [ ]:
import asyncio
import os

import nest_asyncio
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

from autogen import ConversableAgent, LLMConfig
from autogen.mcp import create_toolkit

nest_asyncio.apply()


## Configure LLM

Set your API key. The example uses OpenAI but any AG2-supported model works.


In [ ]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-api-key-here")

llm_config = LLMConfig(
    config_list=[
        {
            "model": "gpt-4o-mini",
            "api_type": "openai",
            "api_key": OPENAI_API_KEY,
        }
    ],
    temperature=0.3,
)


## Connect to TWZRD Agent Intel MCP

The TWZRD Agent Intel server uses the **streamable-HTTP** MCP transport.
We connect directly using `mcp.client.streamable_http` and wrap tools with `create_toolkit`.


In [ ]:
# Target wallet to score (a known active Solana agent)
WALLET = "D1QkbFJKiPsymJ65RKHhF6DFB8sPMfpBaFBzuHKfJGWi"


async def run_trust_check():
    """Connect to TWZRD Agent Intel MCP and run a trust check with AG2."""

    async with streamablehttp_client("https://intel.twzrd.xyz/mcp") as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Wrap MCP tools as AG2-compatible tools
            toolkit = await create_toolkit(session=session)

            # AG2 agent that uses TWZRD tools
            trust_agent = ConversableAgent(
                name="TrustAnalyst",
                llm_config=llm_config,
                system_message=(
                    "You are a Solana agent trust analyst. "
                    "Use the available MCP tools to score a wallet and provide "
                    "an APPROVE or REJECT recommendation for x402 payments."
                ),
            )

            # Register TWZRD tools on the agent
            toolkit.register_for_llm(trust_agent)
            toolkit.register_for_execution(trust_agent)

            # Orchestrator that drives the conversation
            orchestrator = ConversableAgent(
                name="Orchestrator",
                llm_config=False,
                human_input_mode="NEVER",
                max_consecutive_auto_reply=1,
            )

            # Run the trust check
            await orchestrator.a_initiate_chat(
                trust_agent,
                message=(
                    f"Score this Solana agent wallet: {WALLET}\n"
                    "1. Call score_agent to get the trust score\n"
                    "2. Call preflight_check for full due diligence\n"
                    "3. Return APPROVE or REJECT with reasoning"
                ),
                max_turns=3,
            )


asyncio.run(run_trust_check())


## Expected Output

The `TrustAnalyst` agent will:
1. Call `score_agent` → receive trust score (0–1) and payment history
2. Call `preflight_check` → receive full due diligence report
3. Return a structured `APPROVE` or `REJECT` recommendation

Example output:
```
Wallet: D1QkbFJKiPsymJ65RKHhF6DFB8sPMfpBaFBzuHKfJGWi
Trust Score: 0.85
Prior x402 Payments: 48
Recommendation: APPROVE — High repeat-payment agent with 48 confirmed transactions
```

## Links

- TWZRD Agent Intel: https://intel.twzrd.xyz
- MCP endpoint: `https://intel.twzrd.xyz/mcp`
- x402 protocol: https://x402.org
